<a href="https://colab.research.google.com/github/SaiTejaPortfolioDS/Nces-ipeds-analysis/blob/main/Nces_ipeds_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
NCES/IPEDS Postsecondary Education Analytics
Data Analytics Code Sample — Sai Teja Kattiboyina
mvpk240054@gmail.com | linkedin.com/in/saitejakmvp

Purpose:
    Analyze online learning participation, program-level completion outcomes,
    and field-of-study trends across U.S. postsecondary institutions using
    official federal administrative datasets from the National Center for
    Education Statistics (NCES), U.S. Department of Education.

Datasets (IPEDS — Integrated Postsecondary Education Data System):
    1. EFFY2024_dist.csv  — 12-Month Enrollment: Distance Education (AY 2023-24)
    2. CIPCode2020.csv    — Classification of Instructional Programs (CIP 2020)
    3. C2024_A.csv        — Completions Survey by Program and Award Level (AY 2023-24)

Research Questions:
    Q1. How does online (distance) enrollment vary across enrollment level categories?
    Q2. Which CIP program fields have the highest postsecondary completion rates?
    Q3. What is the correlation between distance education participation
        and program completion outcomes at the institution level?
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ─────────────────────────────────────────────
# STEP 1: Load Federal Administrative Datasets
# ─────────────────────────────────────────────

print("Loading IPEDS federal datasets...")

# Distance education enrollment by institution and enrollment level
# EFFYDLEV: 1=Undergraduate, 2=Graduate, 3=First-professional
# EFYDETOT: Total students enrolled exclusively/some distance education
effy = pd.read_csv(
    'Big Data Final Project/EFFY2024_dist.csv',
    encoding='utf-8-sig'
)

# CIP 2020 program metadata: program codes, titles, definitions
cip = pd.read_csv(
    'Big Data Final Project/CIPCode2020.csv',
    encoding='utf-8-sig'
)

# Completions by institution, CIP code, award level, and demographic group
# AWLEVEL: 3=Associate, 5=Bachelor's, 7=Master's, 17=Doctoral
# CTOTALT: Total completions across all students
completions = pd.read_csv(
    'Big Data Final Project/C2024_a.csv',
    encoding='utf-8-sig'
)

print(f"  Enrollment records:  {len(effy):,}")
print(f"  CIP programs:        {len(cip):,}")
print(f"  Completion records:  {len(completions):,}")

# ─────────────────────────────────────────────
# STEP 2: Data Quality Assessment and Cleaning
# ─────────────────────────────────────────────

print("\nAssessing data quality...")

# IPEDS uses suppression flags ('R', 'Z', 'H') in paired columns
# Numeric columns represent actual counts; flag columns indicate suppression status
# Replace suppressed/missing values with NaN for clean analysis

def clean_ipeds_numeric(df, col):
    """Convert IPEDS numeric columns to float, coercing non-numeric flags to NaN."""
    return pd.to_numeric(df[col], errors='coerce')

# Clean enrollment totals
effy['EFYDETOT_clean'] = clean_ipeds_numeric(effy, 'EFYDETOT')
effy['EFYDEEXC_clean'] = clean_ipeds_numeric(effy, 'EFYDEEXC')  # Exclusively distance
effy['EFYDESOM_clean'] = clean_ipeds_numeric(effy, 'EFYDESOM')  # Some distance
effy['EFYDENON_clean'] = clean_ipeds_numeric(effy, 'EFYDENON')  # No distance

# Clean completion totals
completions['CTOTALT_clean'] = clean_ipeds_numeric(completions, 'CTOTALT')

# Report data completeness
total_enroll = len(effy)
missing_enroll = effy['EFYDETOT_clean'].isna().sum()
print(f"  Enrollment — missing values: {missing_enroll:,} / {total_enroll:,} "
      f"({missing_enroll/total_enroll*100:.1f}%)")

total_comp = len(completions)
missing_comp = completions['CTOTALT_clean'].isna().sum()
print(f"  Completions — missing values: {missing_comp:,} / {total_comp:,} "
      f"({missing_comp/total_comp*100:.1f}%)")

# ─────────────────────────────────────────────
# STEP 3: Q1 — Distance Enrollment by Level
# ─────────────────────────────────────────────

print("\nQ1: Analyzing distance enrollment by enrollment level...")

# EFFYDLEV codes: 1=Undergrad, 2=Graduate, 3=First-professional, 99=All levels combined
level_labels = {1: 'Undergraduate', 2: 'Graduate', 3: 'First-Professional', 99: 'All Levels'}

# Aggregate total and exclusive distance enrollment by level
enroll_by_level = (
    effy[effy['EFFYDLEV'] != 99]  # Exclude the "all levels" summary rows
    .groupby('EFFYDLEV')
    .agg(
        total_enrolled=('EFYDETOT_clean', 'sum'),
        excl_distance=('EFYDEEXC_clean', 'sum'),
        some_distance=('EFYDESOM_clean', 'sum'),
        no_distance=('EFYDENON_clean', 'sum'),
        institutions=('UNITID', 'nunique')
    )
    .reset_index()
)

enroll_by_level['level_label'] = enroll_by_level['EFFYDLEV'].map(level_labels)
enroll_by_level['pct_distance'] = (
    (enroll_by_level['excl_distance'] + enroll_by_level['some_distance']) /
    enroll_by_level['total_enrolled'] * 100
).round(1)

print(enroll_by_level[['level_label', 'total_enrolled', 'excl_distance',
                          'pct_distance', 'institutions']].to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(enroll_by_level))
width = 0.35

bars1 = ax.bar(x - width/2, enroll_by_level['excl_distance'] / 1e6,
               width, label='Exclusively Distance', color='#2E6DAD', alpha=0.85)
bars2 = ax.bar(x + width/2, enroll_by_level['some_distance'] / 1e6,
               width, label='Some Distance', color='#4A9E8F', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(enroll_by_level['level_label'], fontsize=11)
ax.set_ylabel('Enrollment (Millions)', fontsize=11)
ax.set_title('U.S. Postsecondary Distance Education Enrollment by Level\n'
             'IPEDS 12-Month Enrollment Survey, AY 2023-24', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Annotate percentage on bars
for i, row in enroll_by_level.iterrows():
    ax.annotate(f"{row['pct_distance']}%\nDistance",
                xy=(i, (row['excl_distance'] + row['some_distance']) / 2e6),
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('Q1_distance_enrollment_by_level.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: Q1_distance_enrollment_by_level.png")

# ─────────────────────────────────────────────
# STEP 4: Q2 — Top CIP Fields by Completions
# ─────────────────────────────────────────────

print("\nQ2: Analyzing completions by CIP program field...")

# Extract 2-digit CIP family code for field-of-study grouping
completions['CIPFamily'] = completions['CIPCODE'].astype(str).str.split('.').str[0].str.zfill(2)

# Aggregate total completions by CIP family and award level
completions_by_field = (
    completions[completions['CTOTALT_clean'] > 0]
    .groupby('CIPFamily')
    .agg(total_completions=('CTOTALT_clean', 'sum'))
    .reset_index()
    .sort_values('total_completions', ascending=False)
)

# Merge with CIP metadata to get program family titles
# CIP codes are stored as ="XX" format — strip the =" and " characters
cip['CIPCode_clean'] = cip['CIPCode'].astype(str).str.replace('="', '').str.replace('"', '').str.strip()
cip['CIPFamily_clean'] = cip['CIPFamily'].astype(str).str.replace('="', '').str.replace('"', '').str.strip()

cip_families = (
    cip[cip['CIPCode_clean'].str.match(r'^\d{2}$')]  # 2-digit family codes only
    [['CIPFamily_clean', 'CIPTitle']]
    .rename(columns={'CIPFamily_clean': 'CIPFamily', 'CIPTitle': 'field_title'})
    .drop_duplicates('CIPFamily')
)

completions_by_field = completions_by_field.merge(cip_families, on='CIPFamily', how='left')
completions_by_field['field_title'] = completions_by_field['field_title'].fillna('Unknown/Unclassified')
completions_by_field['field_short'] = completions_by_field['field_title'].str[:45]

# Top 10 fields
top10 = completions_by_field.head(10)

print(top10[['field_short', 'total_completions']].to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    top10['field_short'][::-1],
    top10['total_completions'][::-1] / 1e6,
    color='#1B3A6B', alpha=0.85
)
ax.set_xlabel('Total Completions (Millions)', fontsize=11)
ax.set_title('Top 10 CIP Program Fields by Total Completions\n'
             'IPEDS Completions Survey, AY 2023-24', fontsize=13, fontweight='bold')

for bar, val in zip(bars, top10['total_completions'][::-1]):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.2f}M', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('Q2_top_cip_fields_completions.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: Q2_top_cip_fields_completions.png")

# ─────────────────────────────────────────────
# STEP 5: Q3 — Institution-Level Correlation
#         Distance Enrollment vs. Completions
# ─────────────────────────────────────────────

print("\nQ3: Correlation between distance enrollment and completion outcomes...")

# Aggregate enrollment at institution level (undergraduate only, all-levels summary)
enroll_inst = (
    effy[effy['EFFYDLEV'] == 1]  # Undergraduate level
    .groupby('UNITID')
    .agg(
        total_enrolled=('EFYDETOT_clean', 'sum'),
        distance_enrolled=('EFYDEEXC_clean', 'sum')
    )
    .reset_index()
)
enroll_inst['pct_distance'] = (
    enroll_inst['distance_enrolled'] / enroll_inst['total_enrolled'] * 100
)

# Aggregate completions at institution level
comp_inst = (
    completions[completions['CTOTALT_clean'] > 0]
    .groupby('UNITID')
    .agg(total_completions=('CTOTALT_clean', 'sum'))
    .reset_index()
)

# Merge and analyze
merged = enroll_inst.merge(comp_inst, on='UNITID', how='inner').dropna()
merged = merged[
    (merged['pct_distance'] >= 0) &
    (merged['pct_distance'] <= 100) &
    (merged['total_completions'] > 0) &
    (merged['total_enrolled'] > 0)
]

print(f"  Institutions with complete data: {len(merged):,}")

# Pearson correlation
r, p_value = stats.pearsonr(merged['pct_distance'], np.log1p(merged['total_completions']))
print(f"  Pearson r (pct_distance vs log_completions): {r:.3f} (p={p_value:.4f})")

# Visualize
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    merged['pct_distance'],
    np.log1p(merged['total_completions']),
    alpha=0.3, s=15, color='#2E6DAD'
)

# Regression line
m, b = np.polyfit(merged['pct_distance'], np.log1p(merged['total_completions']), 1)
x_line = np.linspace(0, 100, 100)
ax.plot(x_line, m * x_line + b, color='#C0392B', linewidth=2,
        label=f'Linear fit (r={r:.2f}, p={p_value:.3f})')

ax.set_xlabel('% Students Enrolled Exclusively in Distance Education', fontsize=11)
ax.set_ylabel('Log(Total Completions + 1)', fontsize=11)
ax.set_title('Distance Education Participation vs. Program Completion Outcomes\n'
             'Institution-Level Analysis, IPEDS AY 2023-24', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('Q3_distance_vs_completions_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved: Q3_distance_vs_completions_correlation.png")

# ─────────────────────────────────────────────
# STEP 6: Summary Table
# ─────────────────────────────────────────────

print("\n" + "="*55)
print("ANALYSIS SUMMARY")
print("="*55)
print(f"Total institutions analyzed:    {merged['UNITID'].nunique():,}")
print(f"Total enrollment records:       {len(effy):,}")
print(f"Total completion records:       {len(completions):,}")
print(f"CIP program fields covered:     {completions['CIPFamily'].nunique()}")
print(f"Correlation (distance vs comp): r = {r:.3f}, p = {p_value:.4f}")
print("="*55)
print("\nOutputs saved:")
print("  Q1_distance_enrollment_by_level.png")
print("  Q2_top_cip_fields_completions.png")
print("  Q3_distance_vs_completions_correlation.png")